# CS116 | LLR → OOF sạch (member mới cho stack v7)

`cs116_llr_score.py` chỉ tạo **feature** LLR (`llr_*.npy`, `lp_*.npy`, `llr_orders_*.npy`), chưa phải OOF. Notebook này biến nó thành
một member đúng chuẩn nhóm (`STACKING.md`), để `cs116_stack_v7.py` nạp qua `@source("team", "cs116-s6e10-oof")`.

**BẢN CHẠY TRÊN MÁY CÁ NHÂN.** Notebook dùng module chung của nhóm trong `src/` (`config.py`, `data.py`, `utils.py`) thay vì viết lại.
- `DATA_DIR`, `OOF_DIR`: lấy từ `src/config.py` (mặc định: `data/`, `oof/` trong repo). Muốn đổi: set biến môi trường trước khi chạy.
- `LLR_DIR`: thư mục chứa output của `cs116_llr_score.py` — sửa ở cell cấu hình.

Cài thư viện: `pip install xgboost scikit-learn scipy pandas numpy jupyter`. Chạy CPU khoảng vài chục phút đến hơn 1 giờ tùy máy (ước lượng).

**Quy tắc chống leak (khớp `STACKING.md`):**
- Fold = `StratifiedKFold(5, shuffle=True, random_state=42)`, dùng `folds.csv` qua `load_data()`.
- LLR đến từ distilgpt2 chỉ fine-tune trên dữ liệu gốc, không dùng nhãn cuộc thi → tính một lần cho train+test là hợp lệ.
- Early stopping dùng 10% tách từ **phần train của fold**, không dùng fold đang chấm điểm.
- Chỉ lưu model gốc (tên không bắt đầu bằng `blend_`, `stack_`, `team_`).

**Đầu ra:** `oof_<NAME>.npy` và `test_<NAME>.npy` trong thư mục `OOF_DIR`.
> **Lưu ý chọn Kernel & GPU:**
> - Hãy chọn kernel / môi trường conda **`cs116`** (đã có sẵn CUDA, PyTorch, XGBoost GPU, LightGBM, CatBoost).
> - Khi chạy trên máy cá nhân có card NVIDIA (RTX 4050), notebook tự động nhận diện `device='cuda'` và chạy trong chưa đầy 1 phút.
> - Sau khi chạy xong, file nộp Kaggle sẽ tự động được lưu tại `submissions/sub_<NAME>.csv`.


In [ ]:
import sys, os

# Thêm src/ vào path để import module chung của nhóm
sys.path.insert(0, os.path.abspath("../src"))

import glob, time, subprocess, warnings
import numpy as np, pandas as pd
import xgboost as xgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

from config import DATA_DIR, OOF_DIR, SUB_DIR, N_FOLDS, SEED, TARGET, ID, CAT_COLS, FEATURES
from data import load_data
from utils import save_preds, N_TRAIN, N_TEST

T0 = time.time()
log = lambda *a: print(f"[{time.time() - T0:6.0f}s]", *a, flush=True)

# ================== CẤU HÌNH ==================
# DATA_DIR, OOF_DIR: lấy từ config.py (mặc định: data/, oof/ trong repo).
# Muốn đổi: set biến môi trường DATA_DIR / OOF_DIR trước khi chạy notebook.
LLR_DIR = os.environ.get("LLR_DIR", str(DATA_DIR / "llr"))  # data/llr/ trong repo
# =====================================================
SMOKE = os.environ.get("SMOKE") == "1"

# ---- cấu hình member ----
NAME = "cs116_llr_xgb_v1"      # tên member; đổi theo quy ước <người>_<model>_<phiên bản> nếu muốn
PERSON = "?"                   # A / B / C, chỉ để in dòng experiments.csv
REFIT = False                  # False: dự đoán bằng model train trên 90% (đúng "Cách a" trong STACKING.md)
                               # True : refit trên 100% phần train của fold với số cây = best * 1.1
RUN_ABLATION = False           # True: chạy thêm bản KHÔNG có LLR để biết LLR có giúp không
MAX_TREES, ES_ROUNDS = (60, 10) if SMOKE else (6000, 200)
assert not NAME.startswith(("blend_", "stack_", "team_")), "tên member không được bắt đầu bằng blend_/stack_/team_"

XGB_PARAMS = dict(learning_rate=0.05, max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.7,
                  reg_lambda=5.0, tree_method="hist", enable_categorical=True, eval_metric="auc",
                  random_state=SEED, n_jobs=os.cpu_count() or 4)
try:
    subprocess.check_output("nvidia-smi", stderr=subprocess.DEVNULL); DEV = "cuda"
except Exception:
    DEV = "cpu"
log("xgboost", xgb.__version__, "| device", DEV, "| SMOKE", SMOKE)

## 1. Dữ liệu, nhãn và fold chung

In [ ]:
train, test = load_data()
y = train[TARGET].to_numpy()
fold = train["fold"].to_numpy()
N, NT = len(train), len(test)
print(f"train {N:,} dòng | test {NT:,} dòng   (kỳ vọng {N_TRAIN:,} / {N_TEST:,})")
print("số dòng mỗi fold:", np.bincount(fold).tolist())

## 2. Nạp feature LLR và kiểm tra thứ tự dòng

`cs116_llr_score.py` đọc `train.csv`/`test.csv` theo đúng thứ tự file, nên có thể ghép theo vị trí. Phép thử an toàn:
AUC của LLR với nhãn phải cao rõ rệt (nếu ~0,5 là lệch dòng).

In [ ]:
def find_npy(fname):
    p = os.path.join(LLR_DIR, fname)
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p}: cần output của cs116_llr_score.py (xin từ đồng đội hoặc tải từ Kaggle) rồi sửa LLR_DIR")
    return p


def llr_frame(split):
    g = lambda n: np.load(find_npy(f"{n}_{split}.npy"))
    orders = g("llr_orders")
    df = pd.DataFrame({"llr": g("llr"), "lp_true": g("lp_true"), "lp_false": g("lp_false")})
    for k in range(orders.shape[1]):
        df[f"llr_o{k}"] = orders[:, k]
    if orders.shape[1] >= 2:
        df["llr_gap"] = orders[:, 0] - orders[:, 1]      # hai thứ tự cột bất đồng bao nhiêu
    return df


LLR_TR, LLR_TE = llr_frame("train"), llr_frame("test")
assert len(LLR_TR) == N and len(LLR_TE) == NT, f"độ dài LLR {len(LLR_TR)}/{len(LLR_TE)} không khớp train/test {N}/{NT}"
assert np.isfinite(LLR_TR.to_numpy()).all() and np.isfinite(LLR_TE.to_numpy()).all(), "LLR có NaN/inf"
LLR_COLS = list(LLR_TR.columns)

auc_llr_raw = roc_auc_score(y, LLR_TR["llr"])
fold_auc = [roc_auc_score(y[fold == k], LLR_TR["llr"][fold == k]) for k in range(N_FOLDS)]
print(f"AUC của LLR thô trên train: {auc_llr_raw:.6f} | theo fold: {np.round(fold_auc, 5).tolist()}")
assert auc_llr_raw > 0.6 or SMOKE, "LLR gần như không có tín hiệu: nhiều khả năng lệch thứ tự dòng so với train.csv"

for c in ("llr", "llr_gap"):
    if c in LLR_TR:
        d = abs(LLR_TE[c].mean() - LLR_TR[c].mean()) / LLR_TR[c].std()
        print(f"drift {c}: |mean_test - mean_train| / std = {d:.4f}" + ("   <-- đáng ngờ" if d > 0.1 else ""))

## 3. Baseline chỉ dùng LLR (rẻ, để xem tín hiệu thuần của LLR)

Nếu AUC nằm trong [0,90; 0,962] thì lưu luôn thành một member (rất khác họ cây). Nếu thấp hơn 0,90, cổng lọc của stack sẽ loại, nên không lưu.

In [ ]:
def cv_logreg(cols):
    oof, tst = np.zeros(N), np.zeros(NT)
    for k in range(N_FOLDS):
        tr, va = fold != k, fold == k
        sc = StandardScaler().fit(LLR_TR.loc[tr, cols])
        lr = LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(LLR_TR.loc[tr, cols]), y[tr])
        oof[va] = lr.predict_proba(sc.transform(LLR_TR.loc[va, cols]))[:, 1]
        tst += lr.predict_proba(sc.transform(LLR_TE[cols]))[:, 1] / N_FOLDS
    return oof, tst


oof_lr, tst_lr = cv_logreg(LLR_COLS)
cv_lr = roc_auc_score(y, oof_lr)
print(f"LogReg chỉ trên feature LLR: CV AUC {cv_lr:.6f}")

## 4. Feature thô + XGBoost, có và không có LLR

Cột chữ → `category` (danh sách giá trị lấy từ train+test, không dùng nhãn). Early stopping theo 10% tách từ phần train của fold.

In [ ]:
# Kiểm tra FEATURES trong config.py khớp cột thực tế
expected = set(train.columns) - {ID, TARGET, "fold"}
assert set(FEATURES) == expected, f"FEATURES thiếu/thừa: {set(FEATURES) ^ expected}"

X_base, XT_base = train[FEATURES].copy(), test[FEATURES].copy()
for c in CAT_COLS:
    cats = sorted(set(X_base[c].dropna().astype(str)) | set(XT_base[c].dropna().astype(str)))
    for d in (X_base, XT_base):
        d[c] = pd.Categorical(d[c].astype("string"), categories=cats)
X_llr, XT_llr = pd.concat([X_base, LLR_TR], axis=1), pd.concat([XT_base, LLR_TE], axis=1)
print(f"{len(FEATURES)} cột thô ({len(CAT_COLS)} category) + {len(LLR_COLS)} cột LLR: {LLR_COLS}")


def make_clf(n_estimators, early=False):
    kw = dict(XGB_PARAMS, device=DEV, n_estimators=n_estimators)
    if early:
        kw["early_stopping_rounds"] = ES_ROUNDS
    return xgb.XGBClassifier(**kw)


def run_cv(X, XT, want_test, tag):
    oof = np.zeros(N)
    tst = np.zeros(len(XT)) if want_test else None
    aucs, iters = [], []
    for k in range(N_FOLDS):
        t0 = time.time()
        tr_idx, va_idx = np.where(fold != k)[0], np.where(fold == k)[0]
        rng = np.random.default_rng(42)
        es_idx = rng.choice(tr_idx, size=len(tr_idx) // 10, replace=False)
        fit_idx = np.setdiff1d(tr_idx, es_idx)
        m = make_clf(MAX_TREES, early=True).fit(X.iloc[fit_idx], y[fit_idx],
                                                eval_set=[(X.iloc[es_idx], y[es_idx])], verbose=False)
        best = int(m.best_iteration) + 1
        n_use = best
        if REFIT:
            n_use = max(10, int(best * 1.1))
            m = make_clf(n_use).fit(X.iloc[tr_idx], y[tr_idx], verbose=False)
        oof[va_idx] = m.predict_proba(X.iloc[va_idx], iteration_range=(0, n_use))[:, 1]
        if want_test:
            tst += m.predict_proba(XT, iteration_range=(0, n_use))[:, 1] / N_FOLDS
        aucs.append(roc_auc_score(y[va_idx], oof[va_idx])); iters.append(best)
        log(f"[{tag}] fold {k}: AUC {aucs[-1]:.6f} | {best} cây | {time.time() - t0:.0f}s")
    return oof, tst, np.array(aucs), iters


oof_x, tst_x, auc_x, it_x = run_cv(X_llr, XT_llr, True, "raw+LLR")
cv_x = roc_auc_score(y, oof_x)
print(f"\nXGB raw+LLR: CV AUC {cv_x:.6f} | fold-mean {auc_x.mean():.6f} ± {auc_x.std():.6f} | số cây {it_x}")

In [ ]:
if RUN_ABLATION:
    _, _, auc_b, it_b = run_cv(X_base, XT_base, False, "raw (không LLR)")
    d = auc_x - auc_b
    print(pd.DataFrame({"fold": range(N_FOLDS), "raw": auc_b, "raw+LLR": auc_x, "delta(1e-6)": d * 1e6}).round(6).to_string(index=False))
    print(f"\nLLR giúp {int((d > 0).sum())}/{N_FOLDS} fold | trung bình {d.mean() * 1e6:+.1f}e-6")
    print("Lưu ý: đây mới là tăng của model đơn. Giá trị với stack phải đo bằng nested CV (3 meta seed, >= 10/15 fold).")

## 5. Lưu member (kiểm tra theo chuẩn nhóm)

In [ ]:
# Kiểm tra AUC range theo chuẩn nhóm rồi lưu bằng save_preds() từ src/utils.py
MIN_AUC, MAX_AUC = 0.90, 0.962

def check_and_save(name, oof, tst, y):
    """Kiểm tra AUC range rồi lưu bằng save_preds()."""
    assert not name.startswith(("blend_", "stack_", "team_"))
    assert np.isfinite(oof).all() and np.isfinite(tst).all(), "NaN/inf"
    auc = roc_auc_score(y, oof)
    status = "OK" if MIN_AUC <= auc <= MAX_AUC else ("QUÁ THẤP: stack sẽ loại (<0,90)" if auc < MIN_AUC else "QUÁ CAO: nghi leak (>0,962)")
    print(f"{name}: CV AUC {auc:.6f} [{status}] | OOF mean {oof.mean():.4f} test mean {tst.mean():.4f}")
    save_preds(name, oof, tst)
    return auc, status


auc_main, st_main = check_and_save(NAME, oof_x, tst_x, y)
if MIN_AUC <= cv_lr <= MAX_AUC:
    check_and_save(NAME.replace("xgb", "lr"), oof_lr, tst_lr, y)
else:
    print(f"LogReg-LLR AUC {cv_lr:.4f} ngoài [{MIN_AUC}; {MAX_AUC}] nên không lưu.")

import datetime
print("\nDòng cho experiments.csv (điền LB sau khi nộp):")
print(f"{datetime.date.today()},{PERSON},{NAME},{auc_main:.6f},,XGB cot tho + LLR distilgpt2 (chi hoc du lieu goc), "
      f"ES 10% trong fold, REFIT={REFIT}; fold-mean {auc_x.mean():.6f}")
# Tạo file CSV nộp Kaggle
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub = pd.DataFrame({ID: sample[ID], TARGET: tst_x})
SUB_DIR.mkdir(exist_ok=True)
out_csv = SUB_DIR / f'sub_{NAME}.csv'
sub.to_csv(out_csv, index=False)
print(f'Đã lưu file nộp Kaggle: {out_csv}')
print(sub.head())


## 6. Độ khác biệt so với OOF đã có (nếu gắn pool vào input)

Member có ích khi **khác** các member hiện có. Spearman với OOF khác gần 0,999 nghĩa là gần như trùng.

In [ ]:
mine = rankdata(oof_x)
files = [f for f in sorted(glob.glob(str(OOF_DIR / "oof_*.npy"))) if os.path.basename(f) != f"oof_{NAME}.npy"][:80]
res = []
for f in files:
    try:
        o = np.load(f)
    except Exception:
        continue
    if o.shape == (N,) and np.isfinite(o).all():
        res.append((os.path.basename(f), np.corrcoef(mine, rankdata(o))[0, 1]))
if res:
    res.sort(key=lambda t: t[1])
    print(f"so với {len(res)} OOF trong input: spearman thấp nhất / trung vị / cao nhất = "
          f"{res[0][1]:.4f} / {np.median([r[1] for r in res]):.4f} / {res[-1][1]:.4f}")
    print("giống nhất:", [(n, round(r, 5)) for n, r in res[-3:]])
else:
    print("không có OOF nào trong input để so sánh (bỏ qua).")

## 7. Bước tiếp theo

1. Kiểm tra bằng repo: `cd src && python blend.py cs116_llr_xgb_v1` (hai file phải nằm trong `oof/`). Rồi upload `oof_<NAME>.npy` và `test_<NAME>.npy` lên thư mục `oof/` trên Drive chung, thêm một dòng vào `experiments.csv` (notebook đã in sẵn) và commit notebook này. Người chạy stack sẽ đưa chúng lên Kaggle.
2. Vá `cs116_stack_v7.py` trước khi chạy (hiện **nguồn `team` bị loại khỏi LR, model sửa phần dư và LGB stacker**, nên member của nhóm không có tác dụng):

```python
# (a) cuối file, hàm load_team: chặn blend/stack lọt vào làm member
for name, o, t, how in npy_pairs(d, "team_"):
    if name.startswith(("team_blend_", "team_stack_")):
        continue
    yield name, o, t, how

# (b) công tắc pool, thay I247 bằng I_USE ở SC/FAM, ZL/ZLT, XP/XPT
TEAM = [i for i, s in enumerate(SRCS) if s == "team"]
I_USE = I247 + (TEAM if os.environ.get("POOL") == "team" else [])
```
3. Chạy `POOL=p247` rồi `POOL=team`, so `F["base"]` (AUC 15 fold) theo cặp fold. Chỉ giữ member nếu tăng ở phần lớn fold và qua cả 3 meta seed.